<a href="https://colab.research.google.com/github/AlvarFeher/TFG/blob/main/LetsEndThis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import matplotlib.pyplot as plt
import networkx as nx

In [3]:
from google.colab import drive
drive.mount('/content/drive')
filename = "/content/drive/MyDrive/TFG/data/filtered_data_good.pkl"
!pip install awkward pandas awkward-pandas

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 871.4/871.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 638.7/638.7 kB 30.2 MB/s eta 0:00:00


In [4]:
import pandas as pd
import pickle
import networkx as nx

# Cargar el dataset
with open(filename, 'rb') as file:
    filtered_df = pickle.load(file)

# Asegurar que las columnas sean listas de Python si vienen en awkward arrays
filtered_df['digitE'] = filtered_df['digitE'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['digitR'] = filtered_df['digitR'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['digitC'] = filtered_df['digitC'].apply(lambda x: x[0] if isinstance(x, list) else x)
filtered_df['MCID'] = filtered_df['MCID'].apply(lambda x: x[0] if isinstance(x, list) else x)

# Ahora estamos incluyendo TODAS las partículas, sin filtrar solo fotones
particles = filtered_df.copy()

print(f"Total de particulas en el dataset: {len(particles)}")
print("Ejemplo de datos:")
print(particles.head())

Total de particulas en el dataset: 13615740
Ejemplo de datos:
                            evtnum  digitE  digitR  digitC   MCID       MCP
entry subentry subsubentry                                                 
0     0        0            303626   5.538    12.0     2.0  130.0  4.110076
               1            303626   5.538    12.0     2.0  130.0  4.110076
               2            303626   5.538    12.0     2.0  130.0  4.110076
               3            303626   5.538    12.0     2.0  130.0  4.110076
               4            303626   5.538    12.0     2.0  130.0  4.110076


In [8]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy.spatial import cKDTree

# -------------------------------
# Optimized function to identify seeds using KDTree
# -------------------------------
def find_seeds_optimized(particles, energy_threshold=50):
    """
    Identify seeds in an event using KDTree for fast neighbor search.
    A digit is a seed if it is the maximum in its 3x3 neighborhood and its energy > energy_threshold.
    Returns a list of tuples: (index, digitR, digitC, digitE, MCID)
    """
    # Extract positions for KDTree; using digitR and digitC.
    positions = particles[['digitR', 'digitC']].values
    # Build KDTree with L-infinity metric to mimic Manhattan neighbor search (neighbors within ±1)
    tree = cKDTree(positions, leafsize=10)
    seeds = []
    # Query radius = 1 using p=np.inf gives all points with max(|dr|,|dc|) <= 1
    for index, row in particles.iterrows():
        r, c, e = row['digitR'], row['digitC'], row['digitE']
        # Get neighbor indices
        neighbors_idx = tree.query_ball_point([r, c], r=1, p=np.inf)
        # Get the maximum energy among these neighbors (vectorized over the neighbor indices)
        max_energy = particles.loc[neighbors_idx, 'digitE'].max()
        if e == max_energy and e > energy_threshold:
            seeds.append((index, r, c, e, row['MCID']))
    # Order seeds in descending order of energy.
    seeds = sorted(seeds, key=lambda x: -x[3])
    return seeds

# -------------------------------
# Other functions remain mostly unchanged.
# You can keep build_graph, update_overlapping_edges, analyze_clusters as before.
# -------------------------------

def build_graph(particles, seeds):
    G = nx.DiGraph()
    # Add seed nodes.
    for seed in seeds:
        idx, r, c, energy, mcid = seed
        G.add_node(idx, row=r, column=c, energy=energy, type='seed', MCID=mcid)

    # Add non-seed (neighbor) nodes and connect each to the closest seed.
    for index, row in particles.iterrows():
        r, c, e = row['digitR'], row['digitC'], row['digitE']
        if index in [s[0] for s in seeds]:
            continue
        # Find the closest seed using Manhattan distance
        closest_seed = min(seeds, key=lambda s: abs(s[1] - r) + abs(s[2] - c))
        seed_idx, sr, sc, s_energy, s_mcid = closest_seed
        G.add_node(index, row=r, column=c, energy=e, type='neighbor', MCID=row['MCID'])
        G.add_edge(index, seed_idx, weight=e / s_energy)
    return G

def update_overlapping_edges(G):
    for node in G.nodes:
        in_edges = list(G.in_edges(node, data=True))
        if len(in_edges) > 1:
            total_energy = sum(G.nodes[src]['energy'] for src, _, _ in in_edges)
            for src, _, edge_data in in_edges:
                seed_energy = G.nodes[src]['energy']
                weight = seed_energy / total_energy if total_energy > 0 else 0
                edge_data['weight'] = weight
    overlap_nodes = [node for node in G.nodes if len(list(G.in_edges(node))) > 1]
    return G, overlap_nodes

def analyze_clusters(G):
    clusters = list(nx.weakly_connected_components(G))
    cluster_analysis = []
    clustered_nodes = {}
    for cluster_id, cluster in enumerate(clusters):
        cluster_nodes = list(cluster)
        cluster_seeds = [node for node in cluster_nodes if G.nodes[node]['type'] == 'seed']
        if len(cluster_seeds) != 1:
            continue
        seed_id = cluster_seeds[0]
        seed_mcid = G.nodes[seed_id]['MCID']
        seed_energy = G.nodes[seed_id]['energy']
        energy_by_particle = {}
        for node in cluster_nodes:
            mcid = G.nodes[node]['MCID']
            energy = G.nodes[node]['energy']
            energy_by_particle[mcid] = energy_by_particle.get(mcid, 0) + energy
        dominant_particle = max(energy_by_particle, key=energy_by_particle.get)
        clustered_nodes[cluster_id] = [
            {
                'node_id': node,
                'energy': G.nodes[node]['energy'],
                'MCID': G.nodes[node]['MCID'],
                'type': G.nodes[node]['type'],
                'row': G.nodes[node]['row'],
                'column': G.nodes[node]['column'],
                'clusterId': cluster_id
            } for node in cluster_nodes
        ]
        cluster_analysis.append({
            'Cluster ID': cluster_id,
            'Cluster Size': len(cluster_nodes),
            'Seed MCID': seed_mcid,
            'Seed Energy': seed_energy,
            'Dominant MCID': dominant_particle,
            'Dominant Energy': energy_by_particle[dominant_particle]
        })
    return cluster_analysis, clustered_nodes

def process_event(event_particles, energy_threshold=50):
    event_particles = event_particles.reset_index(drop=True)
    # Use the optimized seed finder here.
    seeds = find_seeds_optimized(event_particles, energy_threshold)
    if not seeds:
        return None
    G = build_graph(event_particles, seeds)
    G, overlap_nodes = update_overlapping_edges(G)
    cluster_analysis, clustered_nodes = analyze_clusters(G)
    return cluster_analysis, clustered_nodes, G

def process_top_n_events(filtered_df, top_n=10, energy_threshold=50):
    # Get top events based on photon counts
    photon_counts_per_event = filtered_df[filtered_df["MCID"] == 22].groupby("evtnum").size().reset_index(name="Photon Count")
    top_events = photon_counts_per_event.sort_values(by="Photon Count", ascending=False).head(top_n)["evtnum"]

    all_cluster_analysis = []
    all_clustered_nodes = {}

    for evtnum in top_events:
        event_particles = filtered_df[filtered_df["evtnum"] == evtnum]
        result = process_event(event_particles, energy_threshold)
        if result is None:
            continue
        cluster_analysis, clustered_nodes, G = result
        for ca in cluster_analysis:
            if ca['Seed MCID'] == 22:
                ca['evtnum'] = evtnum
                all_cluster_analysis.append(ca)
        for cluster_id, nodes in clustered_nodes.items():
            for node in nodes:
                node['evtnum'] = evtnum
                all_clustered_nodes[node['node_id']] = node
    return all_cluster_analysis, all_clustered_nodes

# -------------------------------
# Main execution block
# -------------------------------
if __name__ == '__main__':
    # Assume filtered_df is already defined, e.g. loaded from CSV.
    # filtered_df = pd.read_csv("your_filtered_data.csv")

    cluster_analysis_list, clustered_nodes_dict = process_top_n_events(filtered_df, top_n=10, energy_threshold=50)

    df_clusters = pd.DataFrame(cluster_analysis_list)
    df_clusters.to_csv("clusters_photon_energy_top10_events_optimized.csv", index=False)
    print("Saved clusters data to clusters_photon_energy_top10_events_optimized.csv")

    df_nodes = pd.DataFrame(list(clustered_nodes_dict.values()))
    df_nodes.to_csv("nodes_with_cluster_ids_top10_events_optimized.csv", index=False)
    print("Saved node-cluster mapping to nodes_with_cluster_ids_top10_events_optimized.csv")


Saved clusters data to clusters_photon_energy_top10_events_optimized.csv
Saved node-cluster mapping to nodes_with_cluster_ids_top10_events_optimized.csv


printear el progreso, por q evento va los datos q genera...
